Hi, we recently shared our bike sales dataset, and I want to understand how the business is performing. I need some insights before our strategy meeting.

Can you analyze the dataset and answer these questions?


Overall Sales Performance

1)What is the total revenue, total profit, and total cost for the dataset?
2)What is our profit margin

Sales Trend

1)Which day generated the highest revenue?
2)Are there any patterns in bike sales?

Customer Demographics

1)Which age group buys the most bikes?
2)Is there a difference in purchases between male and female customers?

Geographic Insights

1)Which country and state have the highest sales?
2)Are there any regions underperforming that we should focus on?

Order Behavior

1)What is the average order quantity per transaction?
2)Are there large bulk purchases affecting revenue?

Profitability Analysis

1)Which products have high revenue but low profit?
2)Are there any products where the cost is too high compared to the price?


Can you create a dashboard showing revenue by month, region, and product category?
United States 

In [65]:
import pandas as pd

In [66]:
df=pd.read_excel("Bikesales.xlsx")

In [67]:
df.head()

,Sales_Order #,Date,Day,Month,Year,Customer_Age,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description,Order_Quantity,Unit_Cost,Unit_Price,Profit,Cost,Revenue
0,261695,2021-12-01,1.0,December,2021,39,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",4.0,1252,2295,4172,5008,9180
1,261695,2021-12-01,1.0,December,2021,44,Adults (35-64),M,United Kingdom,England,Bikes,Mountain Bikes,"Mountain-200 Silver, 42",1.0,1266,2320,1054,1266,2320
2,261697,2021-12-02,2.0,December,2021,37,Adults (35-64),M,United States,California,Bikes,Mountain Bikes,"Mountain-400-W Silver, 46",2.0,420,769,698,840,1538
3,261698,2021-12-02,2.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,"Mountain-400-W Silver, 42",1.0,420,769,349,420,769
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",2.0,0,2295,2086,0,4590


In [68]:
df.shape

(89, 19)

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Sales_Order #        89 non-null     int64         
 1   Date                 89 non-null     datetime64[ns]
 2   Day                  88 non-null     float64       
 3   Month                89 non-null     object        
 4   Year                 89 non-null     int64         
 5   Customer_Age         89 non-null     int64         
 6   Age_Group            88 non-null     object        
 7   Customer_Gender      89 non-null     object        
 8   Country              89 non-null     object        
 9   State                89 non-null     object        
 10  Product_Category     89 non-null     object        
 11  Sub_Category         89 non-null     object        
 12  Product_Description  88 non-null     object        
 13  Order_Quantity       88 non-null     

In [70]:
df.columns=df.columns.str.strip() # we can see space at uc,up,profit,cost. so strip removes spaces
df.columns=df.columns.str.lower() #converts all letter to small which easier for future

In [71]:
df.columns=df.columns.str.replace("sales_order #","sales_order") # sale order # will be replaced by sales_order

In [72]:
df.info() #columns name changes so its better compared to earlier

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   sales_order          89 non-null     int64         
 1   date                 89 non-null     datetime64[ns]
 2   day                  88 non-null     float64       
 3   month                89 non-null     object        
 4   year                 89 non-null     int64         
 5   customer_age         89 non-null     int64         
 6   age_group            88 non-null     object        
 7   customer_gender      89 non-null     object        
 8   country              89 non-null     object        
 9   state                89 non-null     object        
 10  product_category     89 non-null     object        
 11  sub_category         89 non-null     object        
 12  product_description  88 non-null     object        
 13  order_quantity       88 non-null     

In [73]:
df.isnull().sum()

sales_order            0
date                   0
day                    1
month                  0
year                   0
customer_age           0
age_group              1
customer_gender        0
country                0
state                  0
product_category       0
sub_category           0
product_description    1
order_quantity         1
unit_cost              0
unit_price             0
profit                 0
cost                   0
revenue                0
dtype: int64

In [74]:
df.isnull().sum().sum() #4 missing values from entire data

4

In [75]:
df[df.isnull().any(axis=1)] #display the rows with missing value

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
10,261704,2021-12-05,NaN,December,2021,42,Adults (35-64),M,Germany,Nordrhein-Westfalen,Bikes,Mountain Bikes,"Mountain-200 Black, 38",4.0,1252,2295,4172,5008,9180
15,261709,2021-12-06,6.0,December,2021,36,NaN,M,Australia,New South Wales,Bikes,Mountain Bikes,"Mountain-200 Black, 42",1.0,1252,2295,1043,1252,2295
21,261715,2021-12-08,8.0,December,2021,39,Adults (35-64),F,United States,Oregon,Bikes,Mountain Bikes,NaN,2.0,1252,2295,2086,2504,4590
22,261716,2021-12-08,8.0,December,2021,35,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-500 Black, 42",NaN,295,540,245,0,0


# Handling the missing values step
#### step 1 :handling day column's missing value

from above table it shows the columns with missing values and day has 1 missing value, day usaully means for particular day sales order has been recorded and if day same values(if previous next value is same) then between values should also be same, so firstly lrts check previous and next value

In [76]:
df[['day','sales_order']].loc[8:13]

,day,sales_order
8,4.0,261702
9,5.0,261703
10,NaN,261704
11,5.0,261705
12,5.0,261706
13,6.0,261707


from the output i got to know that 10 - 5 (my assumption, and logically true) if previous and next value is 5 then definately 10-5(its data from day 5).
so to fill that i would use ffill method

In [77]:
df['day']=df['day'].ffill() #its fill the nan value with some other value(in this case its 5)

In [78]:
df[['day','sales_order']].loc[10]

day                 5.0
sales_order    261704.0
Name: 10, dtype: float64

Now, Nan becomes 5

In [79]:
df['day'].isnull().sum() #no missing values from day

0

#### step 2 : handling customer age_group's missing value

age group category is filled based on customer age and lets check how age group is grouped

In [80]:
df.groupby("age_group").size()

age_group
Adults (35-64)          47
Young Adults (25-34)    31
Youth (<25)             10
dtype: int64

In [81]:
df.loc[15]

sales_order                            261709
date                      2021-12-06 00:00:00
day                                       6.0
month                                December
year                                     2021
customer_age                               36
age_group                                 NaN
customer_gender                             M
country                             Australia
state                         New South Wales
product_category                        Bikes
sub_category                   Mountain Bikes
product_description    Mountain-200 Black, 42
order_quantity                            1.0
unit_cost                                1252
unit_price                               2295
profit                                   1043
cost                                     1252
revenue                                  2295
Name: 15, dtype: object

In [82]:
#from above output we got to know that the customer age is 36 and he belong to adult group lets fill the missing value of age group
def assign_age_group(age):
    if age<25:
        return "Youth (<25)"
    if age>=25 and age<=34:
        return "Young Adults (25-34)"
    if age>=35 and age<=64:
        return "Adults (35-64)"

df['age_group']=df.apply(
    lambda x:assign_age_group(x['customer_age']) if pd.isna(x['age_group']) else x['age_group'], axis=1)

In [83]:
df.loc[15]

sales_order                            261709
date                      2021-12-06 00:00:00
day                                       6.0
month                                December
year                                     2021
customer_age                               36
age_group                      Adults (35-64)
customer_gender                             M
country                             Australia
state                         New South Wales
product_category                        Bikes
sub_category                   Mountain Bikes
product_description    Mountain-200 Black, 42
order_quantity                            1.0
unit_cost                                1252
unit_price                               2295
profit                                   1043
cost                                     1252
revenue                                  2295
Name: 15, dtype: object

age group has assign correct value based on customer age

In [84]:
df['age_group'].isnull().sum()

0

#### step 3 : Handling missing value for product_description
In this we have to be carefull because exactly we dont know which value to fill in place of NaN

lets analyse unit cost and unit price because unit cost and unit price is set based on the product description firstly lets check price that set based on product description

In [85]:
df.groupby("product_description").size()

product_description
Mountain-100 Black, 38        2
Mountain-100 Black, 48        1
Mountain-100 Silver, 44       1
Mountain-200 Black, 38       13
Mountain-200 Black, 42        7
Mountain-200 Black, 46       15
Mountain-200 Silver, 38      14
Mountain-200 Silver, 42       9
Mountain-200 Silver, 46       4
Mountain-400-W Silver, 38     2
Mountain-400-W Silver, 42     4
Mountain-400-W Silver, 46     6
Mountain-500 Black, 40        2
Mountain-500 Black, 42        2
Mountain-500 Black, 44        1
Mountain-500 Black, 52        1
Mountain-500 Silver, 40       1
Mountain-500 Silver, 42       3
dtype: int64

After analysing the Excel sheet, I got to know that the price is distributed based on color for each version of mountain_bikes, and numbers like(38,48,44) don't matter for the unit_price and costprice(just the version of mountain_bikes and colour price has been set), so let's remove that(38,44)

In [86]:
# this code split the value after comma and str[0] means first value is assigned
df['product_description'] = df['product_description'].str.split(',').str[0] 

In [87]:
df['product_description'] #split has done

0        Mountain-200 Black
1       Mountain-200 Silver
2     Mountain-400-W Silver
3     Mountain-400-W Silver
4        Mountain-200 Black
              ...          
84      Mountain-200 Silver
85      Mountain-200 Silver
86       Mountain-200 Black
87       Mountain-500 Black
88       Mountain-200 Black
Name: product_description, Length: 89, dtype: object

In [88]:
df.groupby("product_description").size()

product_description
Mountain-100 Black        3
Mountain-100 Silver       1
Mountain-200 Black       35
Mountain-200 Silver      27
Mountain-400-W Silver    12
Mountain-500 Black        6
Mountain-500 Silver       4
dtype: int64

In [89]:
df.groupby('product_description')[['unit_price','unit_cost']].nunique()

,unit_price,unit_cost
product_description,,
Mountain-100 Black,1,1
Mountain-100 Silver,1,1
Mountain-200 Black,1,2
Mountain-200 Silver,1,1
Mountain-400-W Silver,2,1
Mountain-500 Black,1,1
Mountain-500 Silver,1,1


In [90]:
df.groupby('product_description')[['unit_price','unit_cost']].first()

,unit_price,unit_cost
product_description,,
Mountain-100 Black,3375,1898
Mountain-100 Silver,3400,1912
Mountain-200 Black,2295,1252
Mountain-200 Silver,2320,1266
Mountain-400-W Silver,769,420
Mountain-500 Black,540,295
Mountain-500 Silver,565,308


From the above, I got to know the actual prices of the product, so let's see the unit price and cost price, and then fill NaN with the correct value

In [91]:
df.loc[21]

sales_order                         261715
date                   2021-12-08 00:00:00
day                                    8.0
month                             December
year                                  2021
customer_age                            39
age_group                   Adults (35-64)
customer_gender                          F
country                      United States
state                               Oregon
product_category                     Bikes
sub_category                Mountain Bikes
product_description                    NaN
order_quantity                         2.0
unit_cost                             1252
unit_price                            2295
profit                                2086
cost                                  2504
revenue                               4590
Name: 21, dtype: object

unit cost = 1252 and unit price is 2295, so from the above output for the NaN place Mountain-200 black is the correct value

Let's check before and after value if any of the values are Mountain-200(black or silver), then we will apply ffill or bfill method

In [92]:
df.loc[19:22]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
19,261713,2021-12-08,8.0,December,2021,19,Youth (<25),F,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-500 Silver,4.0,308,565,1028,1232,2260
20,261714,2021-12-08,8.0,December,2021,30,Young Adults (25-34),F,Canada,British Columbia,Bikes,Mountain Bikes,Mountain-200 Silver,4.0,1266,2320,4216,5064,9280
21,261715,2021-12-08,8.0,December,2021,39,Adults (35-64),F,United States,Oregon,Bikes,Mountain Bikes,NaN,2.0,1252,2295,2086,2504,4590
22,261716,2021-12-08,8.0,December,2021,35,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-500 Black,NaN,295,540,245,0,0


In [93]:
df.loc[
    (df['unit_price'] == 2295) & 
    (df['unit_cost'] == 1252) & 
    (df['product_description'].isna()),
    'product_description'
] = 'Mountain-200 Black'

In [94]:
df.isnull().sum()

sales_order            0
date                   0
day                    0
month                  0
year                   0
customer_age           0
age_group              0
customer_gender        0
country                0
state                  0
product_category       0
sub_category           0
product_description    0
order_quantity         1
unit_cost              0
unit_price             0
profit                 0
cost                   0
revenue                0
dtype: int64

In [95]:
df.loc[21]

sales_order                         261715
date                   2021-12-08 00:00:00
day                                    8.0
month                             December
year                                  2021
customer_age                            39
age_group                   Adults (35-64)
customer_gender                          F
country                      United States
state                               Oregon
product_category                     Bikes
sub_category                Mountain Bikes
product_description     Mountain-200 Black
order_quantity                         2.0
unit_cost                             1252
unit_price                            2295
profit                                2086
cost                                  2504
revenue                               4590
Name: 21, dtype: object

In [96]:
df.groupby("product_description").size() #output changes Mountain-200 black from 35 to 36

product_description
Mountain-100 Black        3
Mountain-100 Silver       1
Mountain-200 Black       36
Mountain-200 Silver      27
Mountain-400-W Silver    12
Mountain-500 Black        6
Mountain-500 Silver       4
dtype: int64

#### step 4 : Handling missing value for the order's quantity columns

In [97]:
df.loc[22]

sales_order                         261716
date                   2021-12-08 00:00:00
day                                    8.0
month                             December
year                                  2021
customer_age                            35
age_group                   Adults (35-64)
customer_gender                          F
country                      United States
state                           California
product_category                     Bikes
sub_category                Mountain Bikes
product_description     Mountain-500 Black
order_quantity                         NaN
unit_cost                              295
unit_price                             540
profit                                 245
cost                                     0
revenue                                  0
Name: 22, dtype: object

In [98]:
# Here, unit_cost is 295, unit_price is 540, and profit is 245,
#lets assume order_quantity as 1
#cost=order_quantity * unit_cost = 1 * 295 = 295
#revenue i= order_quantity*unit_price= 1*540=540
#profit=revenue-cost=540-295=245

#lets assume order_quantity as 2
#cost=order_quantity * unit_cost = 2* 295 = 590
#revenue i= order_quantity*unit_price= 2*540=1080
#profit = revenue-cost = 540- 295 = 490. 490 is not a profit, hence 1 is the quantity'''

In [99]:
df['order_quantity'] = df['order_quantity'].fillna(1) #only one value is missing so im applying 1 directly be carefull while applying for more missing value

In [100]:
df['order_quantity'].isnull().sum()

0

In [101]:
df.loc[[10,15,21,22]]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
10,261704,2021-12-05,5.0,December,2021,42,Adults (35-64),M,Germany,Nordrhein-Westfalen,Bikes,Mountain Bikes,Mountain-200 Black,4.0,1252,2295,4172,5008,9180
15,261709,2021-12-06,6.0,December,2021,36,Adults (35-64),M,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-200 Black,1.0,1252,2295,1043,1252,2295
21,261715,2021-12-08,8.0,December,2021,39,Adults (35-64),F,United States,Oregon,Bikes,Mountain Bikes,Mountain-200 Black,2.0,1252,2295,2086,2504,4590
22,261716,2021-12-08,8.0,December,2021,35,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-500 Black,1.0,295,540,245,0,0


##### some time missing values is not dectectable like it may assign 0 value so assign correct values in place of 0

In [102]:
df[(df['unit_price'] == 0) | (df['unit_cost'] == 0)]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-200 Black,2.0,0,2295,2086,0,4590
8,261702,2021-12-04,4.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-400-W Silver,4.0,420,0,1396,1680,0


In [103]:
import numpy as np
df['unit_price'] = df['unit_price'].replace(0, np.nan)
df['unit_cost'] = df['unit_cost'].replace(0, np.nan)

In [104]:
df['unit_price'] = df.groupby('product_description')['unit_price'] \
                     .transform(lambda x: x.fillna(x.mode()[0]))

df['unit_cost'] = df.groupby('product_description')['unit_cost'] \
                    .transform(lambda x: x.fillna(x.mode()[0]))

In [105]:
df.loc[[4,8]]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-200 Black,2.0,1252.0,2295.0,2086,0,4590
8,261702,2021-12-04,4.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-400-W Silver,4.0,420.0,769.0,1396,1680,0


##### after cross verification values has been set correctly

In [106]:
df[(df['profit'] == 0) | (df['cost'] == 0) | (df['revenue'] == 0)]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-200 Black,2.0,1252.0,2295.0,2086,0,4590
8,261702,2021-12-04,4.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-400-W Silver,4.0,420.0,769.0,1396,1680,0
22,261716,2021-12-08,8.0,December,2021,35,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-500 Black,1.0,295.0,540.0,245,0,0


We have a 0 , instead of replacing, let's calculate revenue, cost, and profit entirely so that even if there are some mistakes, it would make it correct

In [107]:
df['cost']=df['order_quantity']*df['unit_cost']
df['revenue']=df['order_quantity']*df['unit_price']
df['profit']=df['revenue']-df['cost']

In [108]:
df.loc[[4,8,22]]

,sales_order,date,day,month,year,customer_age,age_group,customer_gender,country,state,product_category,sub_category,product_description,order_quantity,unit_cost,unit_price,profit,cost,revenue
4,261699,2021-12-03,3.0,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-200 Black,2.0,1252.0,2295.0,2086.0,2504.0,4590.0
8,261702,2021-12-04,4.0,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,Mountain-400-W Silver,4.0,420.0,769.0,1396.0,1680.0,3076.0
22,261716,2021-12-08,8.0,December,2021,35,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,Mountain-500 Black,1.0,295.0,540.0,245.0,295.0,540.0


In [109]:
df.isnull().sum().sum()

0

In [110]:
# dropping the columns which are unwanted for my analysis

In [111]:
#checking wheather is only december 
df.groupby("month").size()

month
December    88
Decmber      1
dtype: int64

Yes, the dataset is only for December, hence don't need it

In [112]:
df.groupby("year").size()

year
2021    89
dtype: int64

The year is also the same for the entire dataset, as we have date, we don't need it\

In [113]:
df.groupby("product_category").size()

product_category
Bikes    89
dtype: int64

In [114]:
df.groupby("sub_category").size() #keeping it because i need 

sub_category
Mountain Bikes    89
dtype: int64

In [115]:
# dropping all unwanted columns 
df.drop(columns=['month','product_category','year'], inplace=True)

In [116]:
df.columns

Index(['sales_order', 'date', 'day', 'customer_age', 'age_group',
       'customer_gender', 'country', 'state', 'sub_category',
       'product_description', 'order_quantity', 'unit_cost', 'unit_price',
       'profit', 'cost', 'revenue'],
      dtype='object')

In [117]:
df.columns=df.columns.str.replace("sub_category","product_category")

In [118]:
df.columns

Index(['sales_order', 'date', 'day', 'customer_age', 'age_group',
       'customer_gender', 'country', 'state', 'product_category',
       'product_description', 'order_quantity', 'unit_cost', 'unit_price',
       'profit', 'cost', 'revenue'],
      dtype='object')

In [119]:
#checking country and state have space, converting to uppercase
df.groupby('country').size()

country
 United States     1
Australia         27
Canada             6
France             8
Germany            6
United  States     1
United Kingdom     9
United States     30
United States      1
dtype: int64

In [120]:
df['country'] = df['country'].str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()

In [121]:
df.groupby('country').size()

country
AUSTRALIA         27
CANADA             6
FRANCE             8
GERMANY            6
UNITED KINGDOM     9
UNITED STATES     33
dtype: int64

now country column is clean

In [122]:
df.groupby('state').size()

state
British Columbia        6
California             20
England                 9
Hamburg                 1
Hessen                  2
New South Wales        14
Nord                    1
Nordrhein-Westfalen     3
Oregon                  4
Queensland              6
Seine (Paris)           4
Seine Saint Denis       1
Seine et Marne          1
Somme                   1
South Australia         1
Victoria                6
Washington              9
dtype: int64

In [123]:
df['state_group'] = df['state'].replace({
    'Seine (Paris)': 'Paris Region',
    'Seine Saint Denis': 'Paris Region',
    'Seine et Marne': 'Paris Region'
})

In [124]:
df.groupby('state').size()

state
British Columbia        6
California             20
England                 9
Hamburg                 1
Hessen                  2
New South Wales        14
Nord                    1
Nordrhein-Westfalen     3
Oregon                  4
Queensland              6
Seine (Paris)           4
Seine Saint Denis       1
Seine et Marne          1
Somme                   1
South Australia         1
Victoria                6
Washington              9
dtype: int64

In [125]:
 pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [127]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

host = "localhost"
dbname = "sales_analysis"
user = "postgres"
password = "Chaithra@17"
port = "5432"

# URL-encode the password
password_encoded = quote_plus(password)

engine = create_engine(f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{dbname}")

table_name = "bikesales"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data sucessfully loaded into table '{table_name}' in database '{dbname}',")

Data sucessfully loaded into table 'bikesales' in database 'sales_analysis',
